# Optimize FEM's Experiment with Ax


In [123]:
import os
import re
from pathlib import Path

import pandas as pd
from ax.api.client import Client
from ax.api.configs import ChoiceParameterConfig, RangeParameterConfig

In [124]:
NUM_NEW_EXP = 1

In [125]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [126]:
folder_path = Path(os.getcwd()) / "data"

In [127]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)


,trial_index,L_resin,L_metal,L_wood,deformation
0,0,25,25,50,50


In [128]:
# Filter out rows with NaN values in the "deformation" column
filt = exp["deformation"].isna()
exp_todo = exp[filt].copy()
exp = exp[~filt]
display(exp)
display(exp_todo)

,trial_index,L_resin,L_metal,L_wood,deformation
0,0,25,25,50,50


,trial_index,L_resin,L_metal,L_wood,deformation


In [129]:
# 2. Configure where Ax will search.
client.configure_experiment(
    name="parn_exp",
    parameters=[
        RangeParameterConfig(
            name="L_resin",
            bounds=(0, 50),
            parameter_type="float",
        ),
        RangeParameterConfig(
            name="L_metal",
            bounds=(0, 50),
            parameter_type="float",
        ),
        # ChoiceParameterConfig(
        #     name="M",
        #     values=["Metal", "Resin", "Wood"],
        #     parameter_type="str",
        # ),
    ],
)

In [130]:
client.configure_optimization(objective="-deformation")

In [131]:
for idx, row in exp.iterrows():
    # trial_index,L_resin,L_metal,L_wood,deformation
    trial_index = client.attach_trial(
        parameters={
            "L_resin": row["L_resin"],
            "L_metal": row["L_metal"],
        }
    )
    client.complete_trial(
        trial_index=trial_index, raw_data={"deformation": float(row["deformation"])},
    )

c:\Users\admin\Coding\warisa-students\nat_project\.venv\Lib\site-packages\ax\api\client.py:575: RuntimeWarning: Parameterization {'L_resin': np.int64(25), 'L_metal': np.int64(25)} is in out-of-design. Ax will still attach the trial for use in candidate generation.
  _, trial_index = self._experiment.attach_trial(
[INFO 06-25 14:25:41] ax.api.client: Trial 0 marked COMPLETED.


In [134]:
if exp_todo.shape[0] == 0:
    suggest_params = []
    for i in range(NUM_NEW_EXP):
        data = client.get_next_trials(max_trials=1)
        for trial_index, parameters in data.items():
            # print(f"Trial {trial_index} with parameters {parameters}")
            suggest_params.append({"trial_index": trial_index, **parameters})
            # client.complete_trial(
            #     trial_index=trial_index,
            #     raw_data={},
            # )
    _exp = pd.DataFrame(suggest_params)
    _exp["L_wood"] = 100 - _exp["L_resin"] - _exp["L_metal"]
    _exp["deformation"] = pd.NA
    exp = pd.concat([exp, _exp], ignore_index=True).reset_index(drop=True)
    display(exp)
    SAVE_EXCEL = True
else:
    print("There are still experiment to be conducted. Skip generating new sets")
    SAVE_EXCEL = False

[INFO 06-25 14:25:41] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol'), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 06-25 14:25:41] ax.api.client: Generated new trial 1 with parameters {'L_resin': 25.0, 'L_metal': 25.0} using GenerationNode CenterOfSearchSpace.


,trial_index,L_resin,L_metal,L_wood,deformation
0,0,25.0,25.0,50.0,50
1,1,25.0,25.0,50.0,<NA>


In [135]:
if SAVE_EXCEL:
    exp.to_excel(folder_path / "experiments.xlsx", index=False)